# SCD Readability Experiment (TF-IDF)

This notebook implements and evaluates a **Semantic Coherence Distance (SCD)** metric for assessing text readability using TF-IDF embeddings.

## What This Experiment Does

The SCD metric computes the average cosine distance between consecutive sentence embeddings to quantify semantic coherence:
1. Split text into sentences
2. Compute TF-IDF embeddings for each sentence
3. Calculate cosine distances between consecutive sentence pairs
4. Average the distances to get the SCD score

**Hypothesis**: Texts with lower semantic coherence (higher SCD) are harder to read.

## Datasets
- **CLEAR Corpus**: Texts with human readability judgments
- **OneStopEnglish**: Texts in 3 difficulty levels
- **WikiLarge**: Simplification pairs (simple vs normal)

## Output
The notebook computes SCD scores, compares with Flesch-Kincaid grade level, and evaluates correlation with human judgments.

In [ ]:
# Install dependencies - follows aii-colab skill pattern
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Packages NOT pre-installed on Colab (always install)
_pip('loguru')
_pip('textstat')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3', 'matplotlib==3.10.0')

In [ ]:
# Imports - copied from original method.py with minimal additions for notebook
import re
import json
import time
import numpy as np
from pathlib import Path
from typing import Dict, List, Optional
from scipy.stats import pearsonr
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_distances
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import sys

# Try to import loguru, fallback to basic logging
try:
    from loguru import logger
    logger.remove()
    logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")
except:
    import logging
    logger = logging.getLogger(__name__)
    logging.basicConfig(level=logging.INFO, format="%(asctime)s|%(levelname)s|%(message)s")

# Try to import textstat for Flesch-Kincaid
try:
    import textstat
    textstat.set_lang('en')
    HAS_TEXTSTAT = True
except:
    HAS_TEXTSTAT = False
    print("textstat not available, using manual Flesch-Kincaid")

In [ ]:
# Data loading helper - uses GitHub URL with local fallback
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-210829-evaluating-embedding-based-semantic-cohe/main/round-2/experiment-1/demo/mini_demo_data.json"

import json, os

def load_data():
    """Load demo data from GitHub URL with local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        print(f"GitHub load failed: {e}")
    
    # Fallback to local file
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    
    raise FileNotFoundError("Could not load mini_demo_data.json")

# Load the data
print("Loading demo data...")
data = load_data()
print(f"Loaded {len(data['datasets'])} dataset(s)")
for dataset in data['datasets']:
    print(f"  - {dataset['dataset']}: {len(dataset['examples'])} examples")

In [ ]:
# Config cell - ALL tunable parameters set to ABSOLUTE MINIMUM values

# Maximum number of examples to process per dataset (None = all)
# Set to MINIMUM for demo - process all examples in mini dataset
MAX_EXAMPLES = None  # Set to small number like 2 for testing

# TF-IDF parameters
# Maximum number of features (terms) to keep in TF-IDF
# Minimum for demo: 100 (enough to work, faster than 500)
TFIDF_MAX_FEATURES = 100

# Output directory for plots
OUTPUT_DIR = 'plots'

print(f"Config: MAX_EXAMPLES={MAX_EXAMPLES}, TFIDF_MAX_FEATURES={TFIDF_MAX_FEATURES}")

## Helper Functions

These functions implement the core SCD calculation and readability metrics.

In [ ]:
# Tokenize text into sentences
def tokenize_sentences(text: str) -> List[str]:
    """Split text into sentences using punctuation marks."""
    sentences = re.split(r'(?<=[.!?])\\s+', text.strip())
    return [s for s in sentences if s.strip()]

# Compute SCD (Semantic Coherence Distance) for a text
def compute_scd(text: str, max_features: int = 500) -> float:
    """
    Compute Semantic Coherence Distance using TF-IDF embeddings.
    
    SCD = average cosine distance between consecutive sentence embeddings
    Higher SCD = less coherent = potentially harder to read
    
    Args:
        text: Input text to analyze
        max_features: Maximum TF-IDF features (from config)
    
    Returns:
        Float SCD score or np.nan if computation fails
    """
    sentences = tokenize_sentences(text)
    
    # Need at least 2 sentences to compute distances
    if len(sentences) < 2:
        return np.nan
    
    try:
        # Create TF-IDF embeddings for sentences
        tfidf = TfidfVectorizer(max_features=max_features)
        vectors = tfidf.fit_transform(sentences).toarray()
        
        # Compute cosine distances between consecutive sentences
        cos_dists = []
        for i in range(len(vectors) - 1):
            dist = cosine_distances([vectors[i]], [vectors[i+1]])[0][0]
            cos_dists.append(dist)
        
        # Return average distance
        return float(np.mean(cos_dists))
    
    except Exception as e:
        print(f"SCD computation failed: {e}")
        return np.nan

# Compute Flesch-Kincaid readability score
def compute_flesch_kincaid(text: str) -> float:
    """
    Compute Flesch-Kincaid grade level.
    
    Uses textstat if available, otherwise manual calculation.
    Lower grade = easier to read (grade 1-12+)
    """
    if HAS_TEXTSTAT:
        try:
            return textstat.flesch_kincaid_grade(text)
        except:
            pass
    
    # Manual calculation fallback
    words = text.split()
    sentences = tokenize_sentences(text)
    
    if len(sentences) == 0 or len(words) == 0:
        return np.nan
    
    # Count syllables (approximate)
    syllable_count = sum(max(1, len(re.findall(r"[aeiouy]+", w.lower()))) for w in words)
    
    # Flesch-Kincaid formula
    asl = len(words) / len(sentences)  # Average sentence length
    asw = syllable_count / len(words)   # Average syllables per word
    
    return 0.39 * asl + 11.8 * asw - 15.59

print("Helper functions defined.")

## Process Dataset

Compute SCD and Flesch-Kincaid scores for all examples in the dataset.

In [ ]:
# Process all datasets
def process_dataset(dataset: Dict, max_examples: Optional[int] = None, 
                    tfidf_max_features: int = 500):
    """
    Process a dataset and compute readability metrics for each example.
    
    Args:
        dataset: Dataset dict with 'dataset' name and 'examples' list
        max_examples: Maximum number of examples to process (None = all)
        tfidf_max_features: TF-IDF max features from config
    
    Returns:
        List of result dicts with predictions
    """
    examples = dataset["examples"]
    
    # Limit examples if specified
    if max_examples:
        examples = examples[:max_examples]
    
    print(f"Processing {len(examples)} examples from {dataset['dataset']}")
    
    results = []
    for i, ex in enumerate(examples):
        if i % 10 == 0:
            print(f"  Processed {i}/{len(examples)}")
        
        text = ex["input"]
        target = ex["output"]
        
        # Build result dict
        r = {
            "input": text,
            "output": target,
            "dataset": dataset["dataset"]
        }
        
        # Compute SCD score
        r["predict_scd"] = compute_scd(text, tfidf_max_features)
        
        # Compute Flesch-Kincaid score
        r["predict_flesch_kincaid"] = compute_flesch_kincaid(text)
        
        # Copy metadata fields
        for k, v in ex.items():
            if k.startswith("metadata_"):
                r[k] = v
        
        results.append(r)
    
    return results

# Process all datasets in the loaded data
all_results = {}
for dataset in data["datasets"]:
    results = process_dataset(
        dataset,
        max_examples=MAX_EXAMPLES,
        tfidf_max_features=TFIDF_MAX_FEATURES
    )
    all_results[dataset["dataset"]] = results

print("\\nProcessing complete!")
for name, results in all_results.items():
    print(f"  {name}: {len(results)} results")

## Evaluate Results

Compute correlations between predicted scores and human judgments (for CLEAR corpus).

In [ ]:
# Evaluate CLEAR corpus - correlation with human judgments
def evaluate_clear(results: List[Dict]) -> Dict:
    """
    Evaluate SCD and Flesch-Kincaid correlation with human readability judgments.
    
    Args:
        results: List of result dicts from process_dataset
    
    Returns:
        Dict with correlation results for each metric
    """
    print("Evaluating CLEAR corpus")
    
    # Filter valid examples (with numeric output)
    valid = []
    for r in results:
        try:
            t = float(r["output"])
            if not np.isnan(t):
                valid.append(r)
        except:
            pass
    
    print(f"Valid examples: {len(valid)}")
    
    # Compute correlations for each metric
    metrics = ["predict_scd", "predict_flesch_kincaid"]
    correlations = {}
    
    for metric in metrics:
        values, targets = [], []
        
        for r in valid:
            v = r.get(metric)
            if v is not None and not np.isnan(float(v)):
                values.append(float(v))
                targets.append(float(r["output"]))
        
        # Need at least 10 examples for correlation
        if len(values) >= 10:
            try:
                r_val, p_val = pearsonr(values, targets)
                correlations[metric] = {
                    "pearson_r": float(r_val),
                    "p_value": float(p_val),
                    "n": len(values)
                }
                print(f"  {metric}: r={r_val:.4f}, p={p_val:.4f}, n={len(values)}")
            except Exception as e:
                print(f"Correlation failed for {metric}: {e}")
    
    return correlations

# Run evaluation if CLEAR corpus data exists
evaluation_results = {}
if "clear_corpus" in all_results:
    evaluation_results["clear_corpus"] = evaluate_clear(all_results["clear_corpus"])
    print("\\nClear corpus evaluation complete.")
else:
    print("No CLEAR corpus data found.")

## Visualization

Plot SCD and Flesch-Kincaid scores vs human readability judgments.

In [ ]:
# Generate plots comparing predictions to human judgments
def generate_plots(clear_results: List[Dict], output_dir: str = 'plots'):
    """
    Generate scatter plots for CLEAR corpus.
    
    Args:
        clear_results: Results from CLEAR corpus
        output_dir: Directory to save plots
    
    Returns:
        List of plot file paths
    """
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    print("Generating visualizations")
    
    # Filter valid examples
    valid = []
    for r in clear_results:
        try:
            t = float(r["output"])
            if not np.isnan(t):
                valid.append(r)
        except:
            pass
    
    if len(valid) < 3:
        print("Not enough valid examples for plotting")
        return []
    
    plot_files = []
    metrics = ['predict_scd', 'predict_flesch_kincaid']
    
    for metric in metrics:
        values, targets = [], []
        for r in valid:
            v = r.get(metric)
            if v is not None and not np.isnan(float(v)):
                values.append(float(v))
                targets.append(float(r['output']))
        
        if len(values) < 3:
            continue
        
        # Create scatter plot
        plt.figure(figsize=(8, 6))
        plt.scatter(values, targets, alpha=0.5)
        plt.xlabel(metric.replace('predict_', ''))
        plt.ylabel('Human readability judgment')
        
        # Add correlation to title if possible
        try:
            r_val, p_val = pearsonr(values, targets)
            plt.title(f'{metric.replace("predict_", "")} vs Human (r={r_val:.3f})')
        except:
            plt.title(f'{metric.replace("predict_", "")} vs Human')
        
        # Save plot
        plot_file = os.path.join(output_dir, f'{metric}_vs_human.png')
        plt.savefig(plot_file, dpi=150, bbox_inches='tight')
        plt.close()
        plot_files.append(plot_file)
        print(f"  Saved: {plot_file}")
    
    return plot_files

# Generate plots
if "clear_corpus" in all_results:
    plot_files = generate_plots(all_results["clear_corpus"], OUTPUT_DIR)
    print(f"\\nGenerated {len(plot_files)} plot(s)")
else:
    print("No CLEAR corpus data for plotting.")

## Results Summary

Display key results in a readable format and show the plots.

In [ ]:
# Print results summary
print("="*60)
print("SCD READABILITY EXPERIMENT - RESULTS SUMMARY")
print("="*60)

# Show evaluation results
if evaluation_results:
    print("\\n## Correlation with Human Judgments (CLEAR Corpus)")
    print("-"*60)
    for dataset_name, corr in evaluation_results.items():
        print(f"\\n{dataset_name}:")
        for metric, values in corr.items():
            print(f"  {metric}: r={values['pearson_r']:.4f}, p={values['p_value']:.4f}, n={values['n']}")
else:
    print("\\nNo evaluation results (CLEAR corpus data not found)")

# Show sample predictions
print("\\n## Sample Predictions")
print("-"*60)
for dataset_name, results in all_results.items():
    print(f"\\n{dataset_name} (showing first 3):")
    for i, r in enumerate(results[:3]):
        print(f"\\n  Example {i+1}:")
        print(f"    Text: {r['input'][:80]}...")
        print(f"    Human judgment: {r['output']}")
        print(f"    SCD score: {r.get('predict_scd', 'N/A')}")
        print(f"    Flesch-Kincaid: {r.get('predict_flesch_kincaid', 'N/A')}")

In [ ]:
# Display the generated plots
from IPython.display import Image, display

print("Visualizations:")
print("="*60)

if 'plot_files' in locals() and plot_files:
    for plot_file in plot_files:
        print(f"\\nDisplaying: {plot_file}")
        try:
            display(Image(filename=plot_file))
        except Exception as e:
            print(f"  Could not display {plot_file}: {e}")
else:
    print("\\nNo plots generated. Make sure CLEAR corpus data was processed.")

# Also create a simple bar chart of results
if all_results:
    print("\\n" + "="*60)
    print("Summary Statistics by Dataset")
    print("="*60)
    
    for dataset_name, results in all_results.items():
        print(f"\\n{dataset_name}:")
        
        # Compute average SCD and FK
        scd_values = [r.get('predict_scd', np.nan) for r in results]
        fk_values = [r.get('predict_flesch_kincaid', np.nan) for r in results]
        
        scd_valid = [v for v in scd_values if not np.isnan(v)]
        fk_valid = [v for v in fk_values if not np.isnan(v)]
        
        if scd_valid:
            print(f"  Average SCD: {np.mean(scd_valid):.4f} (std: {np.std(scd_valid):.4f})")
        if fk_valid:
            print(f"  Average Flesch-Kincaid: {np.mean(fk_valid):.2f} (std: {np.std(fk_valid):.2f})")